In [ ]:
"""
02_data_cleanup.py

Purpose:
Prepare the raw dataset for further analysis.

Tasks:
- remove unused variables
- remove duplicate variables
- convert numeric columns
- convert date columns
- save cleaned data
"""

In [4]:
from config import DATA_CLEANED, MISSING_THRESHOLD
from utils.load_data_raw import load_data_raw
from utils.convert_comma_to_decimal import convert_comma_to_decimal
from utils.parse_yymmdd_to_date import parse_yyyymmdd
from utils.parse_sas_date import parse_sas_date
from utils.print_section import print_section

# -------------------------------------------------------------------
# Variables to remove
# -------------------------------------------------------------------

COLUMNS_TO_DELETE = [
    'score', 'ratio12', 'ratio14', 'ratio17', 'ratio18', 'ratio19',
    'score_w', 'z_score_rel', 'z_score_tot', 'z_score_rel_w',
    'z_score_tot_w', 'z_addscore', 'z_add',
    'dgrmovescount', 'dgractionpubs', 'dgrstreetfails',
    'dgrstreetstop', 'dgrstreetall', 'dgrpersred',
    'dgrpersgr', 'dgrpersyel', 'dgrpersblue',
    'dgrstreet_ratio', 'total_actions', 'action_rate',
    'afsluit', 'neerlegging', 'obs', 'obser', 'append', 'tag',
    'q_laat_ratio', 'delta_aantal_dagen', 'delta_ad_abs',
    'sum_delta', 'avg_delta', 'aantal_dagen_w',
    'delta_aantal_dagen_w', 'som_laat', 'som_laat_drie',
    'laat_ratio', 'laatstejaar_laat',
    'laatstejaar_laat_dummy', 'avg_delta_w', 'q_avg_delta',
    'avg_delta_pre', 'laat_ratio_dummy', 'laat_dummy_pre',
    'laat_ratio_pre', 'laat_ord', 'fail_dummy_drie',
    'trade_debt', 'fin_debt', 'total_liabilities',
    'lev', 'current_ratio', 'current_ratio_pre',
    'current_dummy', 'net_income_pre', 'loss_dummy',
    'growth', 'lev_w', 'size_w', 'age_w',
    'growth_w', 'trade_debt_w', 'fin_debt_w',
    'move_rate_w', 'fail_afstand',
    'laatste_jaar', 'jvo'
]

# -------------------------------------------------------------------
# Duplicate columns
# -------------------------------------------------------------------

DUPLICATE_COLUMNS = [
    'Rub16',
    'rub16',
    'rub29_58',
    'rub170_4',
    'rub43',
    'rub21',
    'rub9904',
    'rub42_48',
    'rub20_58',
    'rub17_49',
    'rub175',
    'rub44',
    'nace'
]

# -------------------------------------------------------------------
# Date columns
# -------------------------------------------------------------------

DATE_COLUMNS_YYYYMMDD = [
    'startdate',
    'closedate',
    'deposit',
    'oprichting',
    'stopdatum',
    'nglnbb',
    'fail'
]

DATE_COLUMNS_SAS = [
    'faling_datum'
]

# -------------------------------------------------------------------
# Columns excluded from numeric conversion
# -------------------------------------------------------------------

EXCLUDE_COLUMNS = [
    'vat',
    'bookyear',
    'nature',
    'hoofdnacebel',
    'industry',
    'rechtsvorm',
    'stopreden',
    'jaar_van_faling',
    'fail_dummy'
]

# -------------------------------------------------------------------
# Protect target-related columns from missing-value deletion
# -------------------------------------------------------------------

PROTECTED_COLUMNS = [
    'jaar_van_faling',
    'fail',
    'faling_datum',
    'fail_dummy'
]

# -------------------------------------------------------------------
# Load data
# -------------------------------------------------------------------

print_section("Load raw data")

df = load_data_raw()

print(df.shape)
original_rows, original_columns = df.shape # Save original shape for log

# -------------------------------------------------------------------
# Remove unused variables
# -------------------------------------------------------------------

print_section("Remove unused variables")

drop_cols = [
    c for c in COLUMNS_TO_DELETE
    if c in df.columns
]

df = df.drop(columns=drop_cols)
dropped_unused_columns = drop_cols

print(f"Dropped {len(drop_cols)} columns")

# -------------------------------------------------------------------
# Remove duplicate variables
# -------------------------------------------------------------------

print_section("Remove duplicate variables")

drop_cols = [
    c for c in DUPLICATE_COLUMNS
    if c in df.columns
]

df = df.drop(columns=drop_cols)
dropped_duplicate_columns = drop_cols

print(f"Dropped {len(drop_cols)} duplicate columns")

# -------------------------------------------------------------------
# Convert numeric columns
# -------------------------------------------------------------------

print_section("Convert numeric columns")

numeric_cols = [
    c for c in df.columns
    if c not in (
        EXCLUDE_COLUMNS
        + DATE_COLUMNS_YYYYMMDD
        + DATE_COLUMNS_SAS
    )
]

for col in numeric_cols:
    df[col] = convert_comma_to_decimal(df[col])

print(f"Converted {len(numeric_cols)} columns")

# -------------------------------------------------------------------
# Convert date columns
# -------------------------------------------------------------------

print_section("Convert date columns")

for col in DATE_COLUMNS_YYYYMMDD:
    if col in df.columns:
        df[col] = parse_yyyymmdd(df[col])

for col in DATE_COLUMNS_SAS:
    if col in df.columns:
        df[col] = parse_sas_date(df[col])

print("Date conversion complete")

# -------------------------------------------------------------------
# Remove highly missing columns
# -------------------------------------------------------------------

print_section("Remove highly missing columns")

missing_pct = df.isna().mean()

drop_cols = [
    col
    for col in missing_pct[
        missing_pct >= MISSING_THRESHOLD
    ].index
    if col not in PROTECTED_COLUMNS
]

df = df.drop(columns=drop_cols)
dropped_missing_columns = drop_cols

print(
    f"Dropped {len(drop_cols)} columns "
    f"with >= {MISSING_THRESHOLD:.0%} missing values"
)

# -------------------------------------------------------------------
# Final overview
# -------------------------------------------------------------------

print_section("Final dataset")

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns):,}")

# -------------------------------------------------------------------
# Save
# -------------------------------------------------------------------

print_section("Save cleaned data")

df.to_csv(
    DATA_CLEANED,
    index=False
)

print(DATA_CLEANED)

# -------------------------------------------------------------------
# Log the columns that are deleted
# -------------------------------------------------------------------
print_section("Write drop log")

LOG_PATH = "../logs/data_cleaned_logs.txt"

with open(LOG_PATH, "w") as f:

    f.write("DATA CLEANING LOG\n")
    f.write("=" * 60 + "\n\n")

    f.write(
        f"Original dataset: {original_rows:,} rows x "
        f"{original_columns:,} columns\n\n"
    )

    f.write(
        f"Final dataset: {len(df):,} rows x "
        f"{len(df.columns):,} columns\n\n"
    )

    f.write(
        f"MISSING THRESHOLD: {MISSING_THRESHOLD:.0%}\n\n"
    )

    # Unused columns
    f.write(
        f"UNUSED COLUMNS REMOVED "
        f"({len(dropped_unused_columns)})\n"
    )
    f.write("-" * 60 + "\n")

    for col in sorted(dropped_unused_columns):
        f.write(f"{col}\n")

    f.write("\n\n")

    # Duplicates
    f.write(
        f"DUPLICATE COLUMNS REMOVED "
        f"({len(dropped_duplicate_columns)})\n"
    )
    f.write("-" * 60 + "\n")

    for col in sorted(dropped_duplicate_columns):
        f.write(f"{col}\n")

    f.write("\n\n")

    # High missing
    f.write(
        f"HIGH-MISSING COLUMNS REMOVED "
        f"({len(dropped_missing_columns)})\n"
    )
    f.write("-" * 60 + "\n")

    for col in sorted(dropped_missing_columns):
        f.write(f"{col}\n")

print(f"Log written to: {LOG_PATH}")


Load raw data
(4502867, 286)

Remove unused variables
Dropped 71 columns

Remove duplicate variables
Dropped 13 duplicate columns

Convert numeric columns
Converted 185 columns

Convert date columns
Date conversion complete

Remove highly missing columns
Dropped 96 columns with >= 90% missing values

Final dataset
Rows    : 4,502,867
Columns : 106

Save cleaned data
/Users/seymaciftci/Code/bankruptcy-master-thesis/data/processed/data_cleaned.csv

Write drop log
Log written to: ../logs/data_cleaned_logs.txt
